# Model Training
This notebook runs the training pipeline for the Convolutional Autoencoder.

In [6]:
import os
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import sys

# Add src directory to path
sys.path.append('../src')

from data.dataset import MVTecVialDataset
from preprocessing.transforms import get_transforms
from models.autoencoder import ConvAutoencoder
from evaluation.metrics import calculate_auroc

In [7]:
# Load configuration
with open("../configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device(config["training"]["device"] if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [8]:
# Setup data
transform, target_transform = get_transforms(image_size=tuple(config["data"]["image_size"]))

dataset_path = "../" + config["data"]["dataset_path"]

train_dataset = MVTecVialDataset(
    root_dir=dataset_path, 
    split="train", 
    transform=transform,
    target_transform=target_transform
)

val_dataset = MVTecVialDataset(
    root_dir=dataset_path, 
    split="test_public", 
    transform=transform,
    target_transform=target_transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config["data"]["batch_size"], 
    shuffle=True, 
    # Set num_workers=0 for Jupyter on Windows to prevent multiprocessing errors
    num_workers=0
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config["data"]["batch_size"], 
    shuffle=False, 
    # Set num_workers=0 for Jupyter on Windows to prevent multiprocessing errors
    num_workers=0
)

In [9]:
# Setup model
model = ConvAutoencoder(
    in_channels=config["model"]["in_channels"], 
    latent_dim=config["model"]["latent_dim"]
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(
    model.parameters(), 
    lr=config["training"]["learning_rate"], 
    weight_decay=config["training"]["weight_decay"]
)

save_dir = "../" + config["training"]["save_dir"]
os.makedirs(save_dir, exist_ok=True)

In [10]:
# Training Loop
epochs = config["training"]["epochs"]
best_auroc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for images, _, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images = images.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    all_labels = []
    all_scores = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            images = images.to(device)
            outputs = model(images)
            
            # Image-level anomaly score is the MSE error per image
            mse = nn.MSELoss(reduction='none')(outputs, images)
            anomaly_scores = mse.view(mse.size(0), -1).mean(dim=1).cpu().numpy()
            
            val_loss += mse.mean().item() * images.size(0)
            
            all_labels.extend(labels.numpy())
            all_scores.extend(anomaly_scores)
            
    val_loss /= len(val_loader.dataset)
    
    image_auroc = calculate_auroc(np.array(all_labels), np.array(all_scores))
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val AUROC: {image_auroc:.4f}")
    
    if image_auroc > best_auroc:
        best_auroc = image_auroc
        torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
        print(f"--> Saved new best model with AUROC: {best_auroc:.4f}")

Epoch 1/50 [Val]: 100%|██████████| 9/9 [00:04<00:00,  2.10it/s]


Epoch 1/50 - Train Loss: 0.1330 - Val Loss: 0.0339 - Val AUROC: 0.4259
--> Saved new best model with AUROC: 0.4259


Epoch 2/50 [Val]: 100%|██████████| 9/9 [00:04<00:00,  2.12it/s]


Epoch 2/50 - Train Loss: 0.0184 - Val Loss: 0.0151 - Val AUROC: 0.4699
--> Saved new best model with AUROC: 0.4699


Epoch 3/50 [Val]: 100%|██████████| 9/9 [00:04<00:00,  2.01it/s]


...

Epoch 48/50 - Train Loss: 0.0102 - Val Loss: 0.0120 - Val AUROC: 0.4449


Epoch 49/50 [Val]: 100%|██████████| 9/9 [00:07<00:00,  1.26it/s]


Epoch 49/50 - Train Loss: 0.0101 - Val Loss: 0.0120 - Val AUROC: 0.4359


Epoch 50/50 [Val]: 100%|██████████| 9/9 [00:07<00:00,  1.27it/s]

Epoch 50/50 - Train Loss: 0.0102 - Val Loss: 0.0121 - Val AUROC: 0.4525
